In [ ]:
import pandas as pd
from google.colab import files


# ============================================================
# 1. UPLOAD promotions_new.csv
# ============================================================

uploaded = files.upload()


# Tự động tìm file promotions_new.csv
promotion_file = None

for filename in uploaded.keys():
    if "promotion_new" in filename.lower() and filename.lower().endswith(".csv"):
        promotion_file = filename
        break


if promotion_file is None:
    raise FileNotFoundError(
        "Không tìm thấy file promotion_new.csv"
    )


# ============================================================
# 2. ĐỌC FILE
# ============================================================

promotions = pd.read_csv(
    promotion_file,
    low_memory=False
)

print("Số dòng promotion_new:", len(promotions))

display(promotions.head())


# ============================================================
# 3. CHUẨN HÓA TÊN CỘT
# ============================================================

promotions.columns = (
    promotions.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
    .str.strip("_")
)


# ============================================================
# 4. KIỂM TRA CÁC CỘT CẦN CHO DIM_PROMOTION
# ============================================================

required_columns = [
    "promo_id",
    "promo_name",
    "promo_type",
    "discount_value",
    "promo_channel",
    "app_category",
    "stackable_flag"
]


missing_columns = [
    col
    for col in required_columns
    if col not in promotions.columns
]


if missing_columns:
    raise ValueError(
        f"Thiếu các cột: {missing_columns}"
    )


# ============================================================
# 5. TẠO DIM_PROMOTION
#
# Chỉ lấy đúng các thuộc tính có trong StarModel.mmd
# ============================================================

dim_promotion = promotions[
    [
        "promo_id",
        "promo_name",
        "promo_type",
        "discount_value",
        "promo_channel",
        "app_category",
        "stackable_flag"
    ]
].copy()


# ============================================================
# 6. CHUẨN HÓA KIỂU DỮ LIỆU
# ============================================================

# Chuỗi
string_columns = [
    "promo_id",
    "promo_name",
    "promo_type",
    "promo_channel",
    "app_category"
]

for col in string_columns:

    dim_promotion[col] = (
        dim_promotion[col]
        .astype("string")
        .str.strip()
    )


# DECIMAL
dim_promotion["discount_value"] = pd.to_numeric(
    dim_promotion["discount_value"],
    errors="coerce"
)


# BOOLEAN
# Giữ dạng 0/1 để thuận tiện khi load vào database
dim_promotion["stackable_flag"] = pd.to_numeric(
    dim_promotion["stackable_flag"],
    errors="coerce"
).astype("Int64")


# ============================================================
# 7. XÓA DÒNG TRÙNG HOÀN TOÀN
#
# Không xóa promo_id trùng vì:
# cùng một promo_id có thể thuộc nhiều app_category.
# ============================================================

dim_promotion = (
    dim_promotion
    .drop_duplicates()
    .reset_index(drop=True)
)


# ============================================================
# 8. TẠO SURROGATE KEY promo_key
#
# promo_key là khóa chính của DIM_PROMOTION
# ============================================================

dim_promotion.insert(
    0,
    "promo_key",
    range(
        1,
        len(dim_promotion) + 1
    )
)


# ============================================================
# 9. SẮP XẾP ĐÚNG THỨ TỰ CỘT TRONG StarModel.mmd
# ============================================================

dim_promotion = dim_promotion[
    [
        "promo_key",
        "promo_id",
        "promo_name",
        "promo_type",
        "discount_value",
        "promo_channel",
        "app_category",
        "stackable_flag"
    ]
]


# ============================================================
# 10. KIỂM TRA KẾT QUẢ
# ============================================================

print("\n===== DIM_PROMOTION =====")

print("Số dòng:", len(dim_promotion))

print(
    "Số promo_key duy nhất:",
    dim_promotion["promo_key"].nunique()
)

print(
    "Số promo_id duy nhất:",
    dim_promotion["promo_id"].nunique()
)


print("\nMissing value:")

print(
    dim_promotion.isna().sum()
)


print("\nPhân bố app_category:")

print(
    dim_promotion["app_category"]
    .value_counts()
)


display(
    dim_promotion.head(20)
)


# ============================================================
# 11. XUẤT FILE DIM_PROMOTION
# ============================================================

dim_promotion.to_csv(
    "dim_promotion.csv",
    index=False
)


print(
    "\nĐã tạo file dim_promotion.csv"
)

